# Proyecto final: Todo modelo es un voto ponderado
**Análisis y Ciencia de Datos · Facultad de Ingeniería, UNIS · Grupo 2 (Wine Quality, tinto)**

Este notebook corre todo el proyecto de principio a fin a partir de los archivos originales
(`data/wine/winequality-red.csv` y California Housing vía `fetch_california_housing`).
Cada inciso vive en un script de `src/`; aquí se ejecutan en orden y se muestran sus tablas y
gráficas. Las derivaciones y respuestas completas están en `notebooks/explicaciones/`.

Tiempo aproximado de ejecución: 3 minutos (la validación cruzada de la Parte 2 es lo más lento).

In [ ]:
import os, sys, runpy
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

# Trabajar desde la raíz del repositorio
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

TABLAS = RAIZ / "resultados" / "tablas"
GRAFICAS = RAIZ / "resultados" / "graficas"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def correr(modulo):
    """Ejecuta un script como si fuera `python -m modulo` desde la raíz."""
    print(f"\n{'=' * 70}\n>>> python -m {modulo}\n{'=' * 70}")
    runpy.run_module(modulo, run_name="__main__", alter_sys=True)

def tabla(nombre, n=None):
    df = pd.read_csv(TABLAS / nombre)
    display(df if n is None else df.head(n))

def figura(nombre, ancho=850):
    display(Image(filename=str(GRAFICAS / nombre), width=ancho))

## Preparación (0.1 y 0.4)
Partición 80/20 con `random_state=0`, `StandardScaler` ajustado solo con entrenamiento y
desviaciones estándar originales guardadas para la Parte 4.

In [ ]:
correr("src.comunes.datos")
display(pd.read_csv(RAIZ / "data" / "wine" / "particiones" / "wine_desviaciones.csv"))

## Parte 1: Los pesos de cinco modelos (California)
1.1 OLS · 1.2 Ridge (intercepto libre y penalizado) · 1.3 k-NN y árbol · 1.4 Random forest
(sin y con bootstrap) · 1.5 Verificación · 1.6 Proximidad.

In [ ]:
for m in ["src.parte1.pesos_ols", "src.parte1.pesos_ridge",
          "src.parte1.pesos_knn_arbol", "src.parte1.pesos_bosque"]:
    correr(m)

In [ ]:
correr("src.parte1.verificacion")
tabla("1_5_verificacion_california.csv")
tabla("1_5_tabla_fila7_california.csv")

In [ ]:
correr("src.parte1.proximidad")
tabla("1_6_proximidad_fila7_california.csv")

## Parte 2: La geometría de una predicción (Wine)
Hiperparámetros por validación cruzada de 5 pliegues, tres consultas con su ficha de evidencia
($N_{ef}$, $c(x)$ y 5 testigos), pesos negativos de OLS, quintiles de $N_{ef}$ y criterios de
desconfianza.

In [ ]:
correr("src.parte2.cv_modelos")
tabla("parte2_hiperparametros.csv")

In [ ]:
correr("src.parte2.seleccion_consultas")
tabla("parte2_consultas_evidencia.csv")

In [ ]:
correr("src.parte2.pesos_negativos_distancia")
tabla("parte2_pesos_negativos_ols.csv")
figura("parte2_pesos_vs_distancia.png")

In [ ]:
correr("src.parte2.quintiles_nef_mae")
tabla("parte2_quintiles_nef_mae.csv")
tabla("parte2_quintiles_sin_gemelo.csv")

In [ ]:
correr("src.parte2.criterios_desconfianza")
tabla("parte2_criterios_resumen.csv")

## Parte 3: ¿Cuánto cuesta la flexibilidad?
3.1 Grados de libertad efectivos y verificación contra `np.trace(S)` (incluye el caso de
k-NN con filas repetidas) · 3.2 Error contra $\mathrm{tr}(S)$ · 3.4 Simulación del optimismo ·
3.5 Brecha contra optimismo · 3.6 Out-of-bag.

In [ ]:
correr("src.parte3.grados_libertad")
correr("src.parte3.verificar_matrices")
tabla("parte3_verificacion_matrices.csv")

In [ ]:
correr("src.parte3.knn_duplicados")
tabla("parte3_knn_duplicados.csv")

In [ ]:
correr("src.parte3.error_complejidad")
figura("parte3_error_complejidad.png")

In [ ]:
correr("src.parte3.simulacion_optimismo")
correr("src.parte3.resultados_montecarlo")
tabla("parte3_montecarlo_resumen.csv")
figura("parte3_montecarlo_comparacion.png")

In [ ]:
correr("src.parte3.brecha_optimismo")
tabla("parte3_brecha_optimismo.csv")

In [ ]:
correr("src.parte3.comparacion_oob")
tabla("parte3_comparacion_oob.csv")
figura("parte3_comparacion_oob.png")

## Parte 4: La escala también es geometría
4.1 k-NN crudo, estandarizado y en PCA 90 % · 4.2 Vecinos compartidos · 4.3 Árbol con y sin
escala · 4.4 k-means con y sin escala.

In [ ]:
correr("src.parte4.knn_escalas")
tabla("4_1_knn_escalas_wine.csv")
tabla("4_1_dominancia_distancia_wine.csv", n=4)

In [ ]:
correr("src.parte4.vecinos_escalas")
tabla("4_2_vecinos_comunes_wine.csv")

In [ ]:
correr("src.parte4.arbol_escalas")
tabla("4_3_arbol_escalas_wine.csv")

In [ ]:
correr("src.parte4.kmeans_escalas")
tabla("4_4_kmeans_tabla_cruzada_wine.csv")
figura("4_4_kmeans_eleccion_k_wine.png")

## Parte 5: Veredicto
Comparación de los cinco modelos con y sin vinos repetidos. El memo para la bodega está en
[`informe/memo_parte5.md`](../informe/memo_parte5.md).

In [ ]:
correr("src.parte2.comparacion_sin_duplicados")
tabla("parte5_comparacion_sin_duplicados.csv")